# Session 3 — From chat to software

**Week 2 · Session 1 · Big ideas: structured output *and* measured prompting**
**Project version: v0.2 — typed triage + drafted reply + cost report**

Last session you made the model speak in ShopWise's voice. Tonight you make it speak in a form your
**code** can read — and then you find out whether it is any good at it, which is a different question
and a harder one.

One sentence carries the session:

> **Structured output is the interface between a probabilistic component and a deterministic one.**

Take that seriously and the rest follows: the schema, the validators, the failure modes nobody warns
you about, and the number at the end that says whether any of it worked.

**How to read this** — run top to bottom; outputs are committed, so it reads without a key. Model
replies vary between runs; every claim below is about behaviour that holds across runs, and where it
does not, the text says so.

**Today's words:** schema · field · validation · structured output · constrained decoding · ground
truth · accuracy · few-shot. Still no frameworks — session 4.


## 1. Setup

One thing is new before we begin. The cell below reads `settings` out of the project snapshot instead
of hard-coding model IDs in the notebook, because from tonight the project *has* a settings object —
section 4.7 is about why that is a Pydantic lesson and not a housekeeping one.

For now: `settings.MODEL` is a string, and unlike a string you typed here, it came from a file that
checked it.

In [1]:
import json
import subprocess
import sys
import time
from pathlib import Path
from typing import Literal, Optional

import yaml
from google import genai
from google.genai import errors, types
from pydantic import BaseModel, Field, ValidationError

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "project"))    # the v0.2 snapshot, beside this notebook
from config import settings                  # noqa: E402  (path set up on the line above)

client = genai.Client()
TICKETS = yaml.safe_load((settings.DATA_DIR / "tickets.yml").read_text(encoding="utf-8"))
by_id = {t["id"]: t for t in TICKETS}

print(f"{settings.APP_NAME} v{settings.VERSION}  ·  {settings.ENVIRONMENT}")
print(f"model: {settings.MODEL}  ·  key found: {settings.has_key}  ·  {len(TICKETS)} tickets")

ShopWise Support Assistant v0.2.0  ·  development
model: gemini-3.1-flash-lite  ·  key found: True  ·  20 tickets


## 2. Software can't `if` on vibes

Here is v0.1 doing its job perfectly. Your CTO now wants one thing from it, and watch what happens.

In [4]:
ticket = by_id["TCK-009"]
reply = client.models.generate_content(
    model=settings.MODEL,
    contents=f"Reply to this customer:\n\nSubject: {ticket['subject']}\n{ticket['body']}",
).text

# The CTO's ask: "page someone when it's urgent."
try:
    if reply.urgency == "high":
        print("paging on-call")
except AttributeError as err:
    print(f"AttributeError: {err}")
    print("\n`reply` is a string. Strings do not have an urgency.")

AttributeError: 'str' object has no attribute 'urgency'

`reply` is a string. Strings do not have an urgency.


In [5]:
print(reply)

Here are a few options, ranging from professional to empathetic. Choose the one that best fits your brand voice.

### Option 1: Professional & Direct (Best for most businesses)
**Subject:** Re: charged twice

Hi [Customer Name],

Thank you for reaching out. I am very sorry for the frustration caused by this duplicate charge.

I have looked into your account and confirmed that an error occurred, resulting in two charges for your order #[Order Number]. I have initiated a refund for the duplicate charge of $129.00 immediately. 

Please note that it typically takes 3–5 business days for the funds to appear back on your bank statement, depending on your financial institution.

We apologize for the inconvenience and appreciate your patience. Please let me know if there is anything else I can assist you with.

Best regards,

[Your Name]
[Company Name]

***

### Option 2: Short & Action-Oriented
**Subject:** Update regarding your duplicate charge

Hi [Customer Name],

I’m sorry to hear that yo

In [6]:
"urgency" in reply

False

The obvious workaround is worse than the error:

In [3]:
print("does 'urgent' appear in the reply?", "urgent" in reply.lower())
print("\nIf it did, we would be branching on whether the model happened to use one particular")
print("word in a paragraph of prose. That is not a program, it is a coin toss with extra steps.")

does 'urgent' appear in the reply? False

If it did, we would be branching on whether the model happened to use one particular
word in a paragraph of prose. That is not a program, it is a coin toss with extra steps.


**Prose is for humans. Software needs a form, not an essay.**

Tonight has two halves, and they are the same idea seen twice:

1. **Structured output** makes the answer *legal* — a typed object your code can branch on, with the
   possible values fixed in advance.
2. **Measurement** asks whether it is *right* — because a legal answer can still be the wrong one, and
   until you have a number you are guessing.

Legal is cheap: it is a schema, and it takes ten minutes. Right is the rest of the job.

## 3. Structure, or prose?

Not every call should return a form, and reaching for one reflexively is its own mistake. The question
is never "is structured output good" — it is **who reads this output**.

| Ask for structure when… | Because |
|---|---|
| your code will **branch** on it | `if triage.needs_human:` needs a field, not a paragraph |
| it goes in a **database** | columns have types |
| you need to **count** it | "how many refund tickets this week?" |
| it **feeds the next step** | one call's output becomes another's input |
| values must be **valid** | `Literal` makes an invalid category impossible, not merely unlikely |

| Keep prose when… | Because |
|---|---|
| **the prose is the product** | the customer reads the reply — a schema would ruin it |
| a human is the only consumer | you are adding ceremony for nobody |
| the shape is not knowable in advance | forcing a form on an open question loses the answer |

v0.2 has one call of each kind: `triage()` returns a form, `draft_reply()` returns prose. Knowing
which is which is the skill.

> **There is a third shape, and it is not today.** Sometimes you do not want the model to *describe*
> something — you want it to *choose an action*: look up this order, issue this refund. That is a tool
> call, it is built on exactly the machinery below, and it is session 5. Structured output first,
> because a tool call is a structured output with consequences.

## 4. Pydantic, properly

Pydantic is not a detail of this session. It is the library you will use at **every boundary of every
AI application you build** — so this section goes further than "define a class" and covers the parts
that show up in code review.

A Pydantic model is a class that **checks itself**. You describe the shape you want; it refuses
anything that does not fit.

### 4.1 A model that checks itself

In [17]:
class Draft(BaseModel):
    category: Literal["billing", "shipping", "refund"]   # only these three, ever
    urgency: int = Field(ge=1, le=5)                     # 1..5, enforced
    summary: str = Field(description="a brief summary of the draft")
    needs_human: bool = False                            # a default makes it optional


ok = Draft(category="billing", urgency=4, summary="Charged twice")
print(ok)
ok.model_dump()
# print("\ntypes are real:", type(ok.urgency), "| not the string '4'")

category='billing' urgency=4 summary='Charged twice' needs_human=False


{'category': 'billing',
 'urgency': 4,
 'summary': 'Charged twice',
 'needs_human': False}

### 4.2 Reading a `ValidationError` like an engineer

When it refuses, it tells you exactly what and where. That report is a data structure, not a message —
which means your program can act on it, and in section 9 it will.

In [5]:
try:
    Draft(category="complaint", urgency=9, summary="x")
except ValidationError as err:
    print(f"{err.error_count()} problem(s):\n")
    for e in err.errors():
        # loc = which field · type = a stable machine-readable code · msg = for humans
        print(f"  field={e['loc'][0]:<10} type={e['type']:<14} {e['msg'][:60]}")

2 problem(s):

  field=category   type=literal_error  Input should be 'billing', 'shipping' or 'refund'
  field=urgency    type=less_than_equal Input should be less than or equal to 5


`type` is the field to reach for in code: `literal_error`, `greater_than_equal`,
`string_too_long`. It is stable across Pydantic versions in a way the English message is not, so a
retry rule that inspects `e["type"]` keeps working after an upgrade and one that greps the message
does not.

**Two different errors wear the same coat, and confusing them wastes afternoons:**

| | raised by | means | what to do |
|---|---|---|---|
| `ValidationError` | Pydantic, in your process | the data did not fit the shape | fix the data, the schema, or ask the model again |
| `ClientError` / `ServerError` | the API | the call itself failed | section 9 — and only one of them is worth retrying |

### 4.3 Getting data in and out

A model is not a dead end. It converts, both ways — and the direction you are going decides the
method.

In [6]:
row = ok.model_dump()                       # -> dict, Python types (for your code)
wire = ok.model_dump_json()                 # -> str,  JSON  (for an API or a queue)
back = Draft.model_validate_json(wire)      # <- str,  validated on the way IN

print("dict :", row)
print("json :", wire)
print("round trip identical:", back == ok)

# The one that catches people: model_dump() keeps Python objects, which json.dumps() then refuses.
# mode="json" converts them first — dates to strings, enums to values, UUIDs to text.
print('\nmode="json" :', ok.model_dump(mode="json"))

dict : {'category': 'billing', 'urgency': 4, 'summary': 'Charged twice', 'needs_human': False}
json : {"category":"billing","urgency":4,"summary":"Charged twice","needs_human":false}
round trip identical: True

mode="json" : {'category': 'billing', 'urgency': 4, 'summary': 'Charged twice', 'needs_human': False}


**Use `model_validate` at every boundary, not just this one.** A dict from a web request, a row
from a database, a JSON blob from a queue — anywhere data you did not write enters code you are
responsible for, one `model_validate` call turns "I hope this has a category" into "it does, or I know
immediately."

### 4.4 `Field` is more than a type

`Field` carries two very different kinds of information, and telling them apart matters more than it
looks — section 6 shows one of them being quietly ignored by the model.

In [ ]:
class TicketFields(BaseModel):
    # CONSTRAINTS — checked by Pydantic, in your process, every time
    urgency: int = Field(ge=1, le=5)
    summary: str = Field(min_length=1, max_length=120, description="short sumamry for human agent note")
    order_ids: list[str] = Field(default_factory=list)   # never `= []` — see below

    # DESCRIPTION — not checked by anything. It is a message to whoever fills the form in,
    # and when that is a model, it is a prompt.
    sentiment: Literal["calm", "frustrated", "angry"] = Field(
        description="How the customer sounds, not how serious the problem is.")


one = TicketFields(urgency=1, summary="first", sentiment="calm")
two = TicketFields(urgency=1, summary="second", sentiment="angry")
one.order_ids.append("ORD-5001")

print("empty list is the default, and it is a FRESH list each time:")
print("  one:", one.order_ids, " two:", two.order_ids, "<- default_factory kept them apart")

empty list is the default, and it is a FRESH list each time:
  one: ['ORD-5001']  two: [] <- default_factory kept them apart


> **`default_factory=list`, never `= []`.** A mutable default in a plain Python class is created
> once and shared by every instance, so one object's append shows up in all of them — a bug that takes
> an hour to find and thirty seconds to explain. Pydantic protects you here, but the habit is what
> transfers to the rest of your Python.

### 4.5 Validators: the rules a type cannot express

Types say *what shape*. Validators say *what is acceptable* — and they come in two flavours that are
worth naming separately, because reviewers confuse them constantly.

- **Cleaning** — the value is fine, the formatting is not. Strip it, collapse it, lowercase it.
- **Policy** — the value is wrong. Reject it, or correct it, because a rule says so.

`field_validator` sees one field. `model_validator(mode="after")` sees the whole object, which is the
only way to express a rule that spans fields.

In [8]:
from pydantic import field_validator, model_validator


class Triage(BaseModel):
    category: Literal["billing", "refund", "order_status"]
    needs_human: bool
    summary: str

    @field_validator("summary")
    @classmethod
    def _tidy(cls, v: str) -> str:                       # CLEANING
        return " ".join(v.split())                       # collapse newlines and double spaces

    @model_validator(mode="after")
    def _money_needs_a_human(self):                      # POLICY — needs two fields at once
        if self.category in {"billing", "refund"}:
            self.needs_human = True                      # escalate; never de-escalate
        return self


t = Triage(category="refund", needs_human=False, summary="  wants   money\n  back  ")
print("summary :", repr(t.summary))
print("needs_human:", t.needs_human, "<- the model said False. The rule disagreed, and the rule wins.")

summary : 'wants money back'
needs_human: True <- the model said False. The rule disagreed, and the rule wins.


**That last line is the most important idea in the session and it is four lines of code.**

The model is asked for `needs_human` and is usually right. "Usually" is not a control. A prompt is a
request; a validator is a guarantee, and it holds when the model is confused, when it is jailbroken
(session 7), and when someone swaps it for a cheaper one next quarter.

> **Models advise. Code decides.** Write it on something.

Note the direction, too: the rule can only ever *escalate*. A rule that can turn `needs_human` off is
a rule that will one day turn off the one that mattered. This exact validator ships in
`project/schemas.py`.

### 4.6 What the model actually receives

Here is the reveal that makes the rest of the session make sense. Your Pydantic class is not sent to
the model. **Its JSON Schema is** — and you can print it.

In [18]:
print(json.dumps(Draft.model_json_schema(), indent=2))

{
  "properties": {
    "category": {
      "enum": [
        "billing",
        "shipping",
        "refund"
      ],
      "title": "Category",
      "type": "string"
    },
    "urgency": {
      "maximum": 5,
      "minimum": 1,
      "title": "Urgency",
      "type": "integer"
    },
    "summary": {
      "description": "a brief summary of the draft",
      "title": "Summary",
      "type": "string"
    },
    "needs_human": {
      "default": false,
      "title": "Needs Human",
      "type": "boolean"
    }
  },
  "required": [
    "category",
    "urgency",
    "summary"
  ],
  "title": "Draft",
  "type": "object"
}


Read what is in there: the field names, the types, the enum values, and — if you wrote them —
your `description` strings. All of it is text the model sees.

Three consequences, and they run the rest of this notebook:

1. **`Field(description=…)` is prompt engineering.** It is the cheapest you will ever do: it sits next
   to the field it describes, so it cannot be misread as being about a different field, and it costs
   nothing to write.
2. **Your field names are prompt engineering too.** `needs_human` teaches the model something;
   `flag_2` teaches it nothing. Name fields the way you would name them to a new colleague.
3. **All of it is sent, on every single call.** Whether you are *billed* for it is a separate
   question with a provider-specific answer — section 10.1 measures it on Gemini and the result is
   not the obvious one. Either way it is text the model must read, and attention is finite even
   where tokens are free.

### 4.7 The other boundary: your own configuration

Pydantic guards the door the model comes through. There is a second door, and it is the one that
actually takes production down at 2am.

<img src="figures/two-boundaries.png" width="820"
     alt="Pydantic standing at both of a program's doors: settings validate what arrives from the
     environment, and a response schema validates what arrives from the model, so everything inside
     the program is already typed.">

Session 2's `config.py` was module-level constants read with `os.getenv`. That is the right amount of
machinery for three settings and one developer. It stops being right the moment a setting is not a
string:

```python
USE_FEW_SHOT = os.getenv("USE_FEW_SHOT", "false")   # -> the STRING "false"
if USE_FEW_SHOT:                                     # -> always True. Always.
```

That switch is stuck on, it will never raise, and you will find it in a token bill. So v0.2's
configuration is a Pydantic model — same library, same day, other boundary.

In [10]:
print(f"{settings.APP_NAME} v{settings.VERSION}  [{settings.ENVIRONMENT}]")
for name, value in settings.model_dump().items():
    print(f"  {name:20} {value!r}")

ShopWise Support Assistant v0.2.0  [development]
  APP_NAME             'ShopWise Support Assistant'
  VERSION              '0.2.0'
  ENVIRONMENT          'development'
  DEBUG                False
  GEMINI_API_KEY       SecretStr('**********')
  MODEL                'gemini-3.1-flash-lite'
  MODEL_STRONG         'gemini-3.6-flash'
  MODEL_SEARCH         'gemini-3.5-flash-lite'
  MODEL_PRO            'gemini-2.5-pro'
  EMBED_MODEL          'gemini-embedding-2'
  MAX_OUTPUT_TOKENS    1024
  REQUEST_TIMEOUT_S    60.0
  MAX_RETRIES          3
  USE_FEW_SHOT         False
  DATA_DIR             PosixPath('/Users/user/Desktop/grow_with_data/ai-llm-app-builder/labs/data')


Two details in that output worth stopping on:

- `USE_FEW_SHOT` is a real `bool`, because a type said so. `SHOPWISE_USE_FEW_SHOT=false` in `.env`
  arrives as `False`, not as a truthy string.
- `GEMINI_API_KEY` printed as `'**********'`. It is a `SecretStr`, so it will not leak into a log
  line, a traceback, or a screen share. Reading it takes a deliberate `.get_secret_value()` — the
  point being that the only way to see the key is to write code that says you meant to.

And configuration errors now arrive **at startup**, naming the field, instead of two hundred lines
later inside somebody's SDK:

In [11]:
from config import Settings

try:
    Settings(ENVIRONMENT="prod")          # not one of the three allowed values
except ValidationError as err:
    for e in err.errors():
        print(f"  {e['loc'][0]}: {e['msg']}")

print("\nThis is the same ValidationError from section 4.2, doing the same job at the other door.")

  ENVIRONMENT: Input should be 'development', 'staging' or 'production'

This is the same ValidationError from section 4.2, doing the same job at the other door.


> **Why the `SHOPWISE_` prefix.** Every field is read from `SHOPWISE_<FIELD>`. `MODEL` and `DEBUG`
> are names half the software on your machine also wants; an unprefixed setting is a variable you will
> one day inherit from a shell you forgot you exported in, on a Friday. `GEMINI_API_KEY` is the
> deliberate exception — it keeps the name Google's SDK looks for, so there is one key in your `.env`
> rather than two that have to agree.

Open `project/config.py` when you get a moment. The field list *is* the documentation: read it top to
bottom and you know everything this program can be told to do.

## 5. The schema staircase

Same discipline as session 2's prompt staircase: one ticket, one variable changed per step, and the
failure shown before the fix.

<img src="figures/schema-staircase.png" width="860"
     alt="Four ascending steps of a growing schema: a string summary, then a free-text category whose
     spelling drifts between runs, then a Literal that closes the value space, then the full typed
     result the code can act on.">

The call shape is new, so read it once before the staircase runs.

In [ ]:
def ask_schema(schema, ticket, runs: int = 1):
    """Send a ticket, get back an instance of `schema`. Runs it `runs` times."""
    out = []
    for _ in range(runs):
        r = client.models.generate_content(
            model=settings.MODEL,
            contents=f"Triage this ticket.\n\nSubject: {ticket['subject']}\n{ticket['body']}",
            config=types.GenerateContentConfig(
                response_mime_type="application/json",   # "answer in JSON"
                response_schema=schema,                  # "...shaped like THIS"
            ),
        )
        out.append(r.parsed)          # .parsed is a real, validated Python object
    return out


tk = by_id["TCK-009"]
print(f"{tk['id']}: {tk['subject']}")

TCK-009: charged twice


> ⚠️ **The trap that will cost you twenty minutes.** Google's docs also show `response_format=`.
> That is an **Interactions API** field and raises `ValidationError: extra_forbidden` here — you met
> it in session 2 section 4. On `generate_content` it is `response_mime_type` + `response_schema`,
> read back through `.parsed`.

### Step 0 — a string in a box

In [20]:
class S0(BaseModel):
    summary: str

print(ask_schema(S0, tk)[0])

summary='Customer reported a double charge of 129.00 for a single order and is requesting a refund for the duplicate transaction.'


Typed, but useless: we have moved prose into a field. Nothing can branch on it yet.

### Step 1 — add a free-text category, and run it three times

In [ ]:
class S1(BaseModel):
    summary: str
    category: str          # free text — the model picks the words

for r in ask_schema(S1, tk, runs=3):
    print(f"  category={r.category!r}")

if r.category == "billing":
    # take some action

  category='Billing'
  category='Billing'
  category='Billing'


**Look at what came back, and then look at it again.**

Depending on the run you will see one of two things, and *both* are the same problem:

- three different answers — `billing`, `Billing Issue`, `Double charge`; or
- the same answer three times, but not the one your code is waiting for. The run committed here
  returned `'Billing'`, with a capital B, three times out of three.

The second is the more dangerous, because it looks stable. It is not stable — it is *this model, this
week*, and you did not choose that capital letter. Now write the code that consumes it:

```python
if triage.category == "billing":   # False. Every time. Silently.
```

No exception, no warning: the branch simply never runs, and the ticket goes wherever your `else`
sends it. You cannot `==` against a vocabulary the model is choosing. This is not the model being
sloppy — you asked for a string and it gave you a string.

### Step 2 — `Literal` pins the vocabulary

In [15]:
class S2(BaseModel):
    summary: str
    category: Literal["billing", "complaint", "order_status", "other",
                      "product_question", "refund", "return", "shipping", "warranty"]

for r in ask_schema(S2, tk, runs=3):
    print(f"  category={r.category!r}")

  category='billing'
  category='billing'
  category='billing'


Three runs, one spelling, every time. The value space is now **closed** — and section 6
explains why that is a guarantee from the decoder rather than good luck.

> **Where these nine came from.** They are the hand-labelled ground truth in `labs/data/tickets.yml`,
> fixed by `docs/adr/0003`. Session 1 section 7.1 showed a shortened five-value list to keep the very
> first demo small; section 7.2 of that same notebook uses the full nine. **The nine are the real
> ones** — they are what we score against in section 8.

### Step 3 — the fields that make it useful

In [16]:
class TriageResult(BaseModel):
    category: Literal["billing", "complaint", "order_status", "other",
                      "product_question", "refund", "return", "shipping", "warranty"]
    urgency: Literal["low", "medium", "high"]
    sentiment: Literal["calm", "frustrated", "angry"]
    needs_human: bool = Field(description="True if money moves, or they ask for a person.")
    summary: str = Field(description="One sentence, under 20 words, for a dashboard row.")


result = ask_schema(TriageResult, tk)[0]
print(result, "\n")

# And now the CTO's line from section 2 finally runs:
if result.needs_human:
    print(f"-> {tk['id']} routed to a human ({result.category}, {result.urgency})")

category='billing' urgency='high' sentiment='calm' needs_human=True summary='Customer requesting a refund for a duplicate charge on their credit card.' 

-> TCK-009 routed to a human (billing, high)


| step | added | what it bought |
|---|---|---|
| 0 | `summary: str` | prose in a box — nothing to branch on |
| 1 | `category: str` | a field, but the values drift between runs |
| 2 | `category: Literal[...]` | **the value space is closed** — `==` is now safe |
| 3 | urgency · sentiment · needs_human | a form the program can act on |

This `TriageResult` is the most durable object in the course: session 10 makes it the graph's triage
output, session 13 makes it the API response model. **Its field names do not change.**

The shipped version in `project/schemas.py` is this class plus the two validators from section 4.5.
Worth opening now — it is forty lines and you have just written most of them.

## 6. What `response_schema` actually does

You have been passing `response_schema=` since section 5. There are three ways to get JSON out of a
model, and they give you three different guarantees. Knowing which one you have is the difference
between a bug you find in a test and one you find in a quarterly report.

<img src="figures/three-ways-to-json.png" width="860"
     alt="The three ways a program gets JSON out of a model — asking in the prompt and parsing,
     constrained decoding from a schema, and a tool call — with what each one actually guarantees and
     what can still go wrong.">

### 6.1 Ask nicely, then `json.loads()`

The way everyone starts: describe the JSON in the prompt and parse the reply.


In [17]:
def as_block(t):
    return f"<ticket>\nSubject: {t['subject']}\n{t['body']}\n</ticket>"


block = as_block(tk)                      # TCK-009, used for the rest of the notebook

# Two tickets: one unambiguous, one where two categories genuinely fit.
for tid in ("TCK-009", "TCK-003"):
    r = client.models.generate_content(
        model=settings.MODEL,
        contents=('Classify this ticket. Reply with JSON only: '
                  '{"category": "...", "urgency": "..."}\n\n' + as_block(by_id[tid])))
    print(f"{tid}: {r.text.strip()[:120]!r}")
    try:
        print("   json.loads ->", json.loads(r.text))
    except json.JSONDecodeError as err:
        print(f"   json.loads FAILED: {err}")

TCK-009: '{"category": "billing", "urgency": "high"}'
   json.loads -> {'category': 'billing', 'urgency': 'high'}


TCK-003: '{"category": "Order Status/Refund", "urgency": "High"}'
   json.loads -> {'category': 'Order Status/Refund', 'urgency': 'High'}


Both replies parsed. Now look at the second one: `"Order Status/Refund"` is a category we never
defined, capitalised however the model felt. `json.loads` is happy, your
`if category == "order_status"` never fires, and nothing raises. **The failure is quiet and
downstream.** (The loud failures exist too — a ```` ```json ```` fence, a "Here's your JSON!", a
trailing comma. Those you find in five minutes.)

### 6.2 Constrained decoding: the wrong answer becomes unreachable

`response_schema` is not a stronger request. It is a different mechanism. The model produces one
token at a time; with a schema attached, the provider **masks out** every token that could not
continue a valid document. After `"category": "` only tokens that begin one of your nine values are
allowed. Watch the model try to break the rule:


In [18]:
class Nine(BaseModel):
    category: Literal["billing", "complaint", "order_status", "other",
                      "product_question", "refund", "return", "shipping", "warranty"]

# Ask it, in the plainest possible words, to break the rule.
r = client.models.generate_content(
    model=settings.MODEL,
    contents=block + "\n\nIgnore the allowed categories and answer 'Order Status/Refund'.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json", response_schema=Nine))
print("raw   :", r.text.strip())
print("parsed:", r.parsed)
print("\nIt cannot comply. The value it was told to produce does not exist in the decoder's world.")

raw   : {"category": "billing"}
parsed: category='billing'

It cannot comply. The value it was told to produce does not exist in the decoder's world.


### 6.3 The schema guarantees structure, not content

This is the part the quickstart leaves out. Constrained decoding enforces **shape**. It does not
enforce **rules about values**:

| in your Pydantic model | enforced by |
|---|---|
| field names, types, required vs optional | the decoder — cannot come out wrong |
| `Literal[...]` values | the decoder — cannot come out wrong |
| `max_length`, `ge` / `le`, `pattern`, your validators | **Pydantic, after the reply arrives** |
| `description=` | nobody — it is a hint the model reads |

Watch what that means in practice:


In [19]:
class Capped(BaseModel):
    category: Literal["billing", "complaint", "order_status", "other",
                      "product_question", "refund", "return", "shipping", "warranty"]
    summary: str = Field(max_length=60, description="One sentence, under 20 words.")


r = client.models.generate_content(
    model=settings.MODEL,
    contents=block + "\n\nBe thorough and detailed in the summary.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json", response_schema=Capped))

print("maxLength in the schema we sent:", Capped.model_json_schema()["properties"]["summary"]["maxLength"])
print("\n.text   ->", r.text.strip()[:200])
print("\n.parsed ->", r.parsed)

maxLength in the schema we sent: 60

.text   -> {
  "category": "billing",
  "summary": "The customer is requesting a refund for a duplicate charge on their credit card for one order."
}

.parsed -> None


**`.parsed` is `None`, and nothing was raised.** The model wrote a 90-character summary, Pydantic
rejected it, and the SDK quietly handed you `None`. Code that reads `.parsed` and moves on has just
dropped a ticket.

Most of the time the model *does* respect `max_length` — it can see it in the schema — which is
exactly what makes this dangerous: it works on every ticket you test and fails on the one you did
not. Validate yourself and you get the real story:


In [20]:
try:
    Capped.model_validate_json(r.text)
    print("  this run happened to fit — run the cell above again; usually it does not.")
except ValidationError as err:
    for e in err.errors():
        print(f"  {e['loc'][0]}: [{e['type']}] {e['msg']}")

print("\nRules to take from this:")
print("  1. never use `.parsed` without checking it for None")
print("  2. content constraints are YOUR job to enforce, and yours to recover from (section 9)")
print("  3. put the limit in the description too — it is the only version the model reads")

  summary: [string_too_long] String should have at most 60 characters

Rules to take from this:
  1. never use `.parsed` without checking it for None
  2. content constraints are YOUR job to enforce, and yours to recover from (section 9)
  3. put the limit in the description too — it is the only version the model reads


### 6.4 Providers accept different subsets of JSON Schema

One more thing you would otherwise discover at 11pm:


In [22]:
from pydantic import ConfigDict


class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")   # -> "additionalProperties": false
    category: Literal["billing", "refund", "other"]


try:
    client.models.generate_content(
        model=settings.MODEL, contents=block,
        config=types.GenerateContentConfig(
            response_mime_type="application/json", response_schema=Strict))
except errors.ClientError as err:
    print(f"{type(err).__name__} {err.code} {err.status}")
    print(f"  {err.message[:160]}")

ClientError 400 INVALID_ARGUMENT
  Invalid JSON payload received. Unknown name "additional_properties" at 'generation_config.response_schema': Cannot find field.


`extra="forbid"` is good Pydantic hygiene, and Gemini rejects the schema it produces. Other providers
reject other things (`$ref`, `oneOf`, deep nesting, `format:`). The portable habit:

> **Keep response schemas boring.** Flat objects, primitive fields, `Literal`s, lists of one type.
> Save clever type modelling for classes that never leave your program.


## 7. Designing a schema the model can fill in well

A database schema is judged on integrity. A schema for a model is judged on one thing: **does its
shape make the right answer easy to produce?** Three rules cover most of it.

### 7.1 Field order is writing order

<img src="figures/field-order.png" width="860"
     alt="Two versions of the same schema side by side, showing that a model writes fields in
     declaration order: putting the reasoning field after the decision only rationalises it, while
     putting it first lets it inform the decision.">

The model writes the JSON top to bottom, in the order you declare the fields. A field can only be
informed by the fields **above** it. That is why `TriageResult` puts the decisions (`category`,
`urgency`, `needs_human`) first and `summary` last: the summary is easy once the decisions are made.

(A popular claim says a `reasoning` field declared *before* the decision improves accuracy. It is
testable — appendix G tests it, and on this task it did not help.)

### 7.2 Leave the model a way to say "not here"

`Literal` forces one of nine answers, including when the model has no idea. If a field may genuinely
be absent, make it `Optional` and say so in the description — otherwise the model will invent
something plausible:


In [25]:
class WithEscapeHatch(BaseModel):
    category: Literal["billing", "complaint", "order_status", "other",
                      "product_question", "refund", "return", "shipping", "warranty"]
    order_id: Optional[str] = Field(
        default=None,
        description="The ORD-XXXX mentioned in the ticket, exactly as written. "
                    "null if the ticket does not mention one. Never invent one.")


for tid in ("TCK-003", "TCK-006"):
    t = by_id[tid]
    r = client.models.generate_content(
        model=settings.MODEL,
        contents=f"<ticket>\nSubject: {t['subject']}\n{t['body']}\n</ticket>",
        config=types.GenerateContentConfig(
            response_mime_type="application/json", response_schema=WithEscapeHatch))
    print(f"{tid}: {r.parsed}")

TCK-003: category='order_status' order_id='ORD-5002'


TCK-006: category='warranty' order_id=None


`None` where there is no order, the real ID where there is one. Without the `Optional`, the model
would have had to write *something* — an invented order number that a later tool would then look up.

### 7.3 The checklist

| smell | fix |
|---|---|
| `category: str` — values drift between runs | `Literal[...]` |
| a field no code reads — output tokens on every call | delete it |
| a field only *your* code should set (timestamps, IDs) | keep it out of the schema; add it in code afterwards (appendix H) |
| no way to say "unsure" or "absent" | `Optional[...] = None`, or an `other` value |
| `max_length` with no description — the model never sees the cap | say the limit in words too |
| vague names: `flag_2`, `val` | name it for a colleague — the name is part of the prompt |
| asking for a date or a total | compute it in code; models are bad at arithmetic and have no clock |

> The schema is read by a language model. Write it the way you would brief a fast, literal new
> colleague who never asks a follow-up question.


## 8. Legal, or right?

`Literal` guarantees the answer is one of nine. **It does not guarantee it is the correct one.** Every
ticket in `tickets.yml` has a hand-written `category`, so we can check.

<img src="figures/measurement-loop.png" width="860"
     alt="The measurement loop — score against hand labels, change one thing, score again — beside a
     panel naming the ceiling that no prompt can lift, because two of the twenty labels are
     arguable.">

### 8.1 Score it

`score_triage.py` (beside this notebook, not inside `project/`) runs the project's `triage()` over
all twenty tickets and compares the answers with the hand labels. It also prints two numbers accuracy
hides: how many replies were **unusable**, and what the run **cost**.


In [26]:
done = subprocess.run([sys.executable, "score_triage.py"],
                      cwd=HERE, capture_output=True, text=True, timeout=900)
print(done.stdout or done.stderr[-1500:])


WITHOUT examples: 18/20 = 90%
  0 unusable response(s) · 1100 output tokens · $0.0028 for the run ($0.14 per 1,000 tickets)

  ticket    hand label         assistant said
  TCK-003   complaint          order_status
  TCK-005   return             warranty

  commonest confusions: complaint -> order_status (x1), return -> warranty (x1)

Read the misses before you change anything. Several are arguable,
and an arguable label is a convention to teach, not a bug to fix.



Read the misses before changing anything. `TCK-003` is an angry message about an order that has not
shipped: labelled `complaint`, called `order_status`. Both are defensible. ShopWise's rule is *"if
our own conduct is the problem, it is a complaint"* — a house convention, not a fact about English.
`TCK-005` (`return` vs `warranty`) is the same kind of disagreement.

So the score measures **"does the model agree with our labelling convention"**, and about two of the
twenty points cannot be won by prompting. Knowing where that ceiling is saves you a week chasing the
last 10%.

### 8.2 Teaching the convention: write the rule, or show examples

**Write the rule.** `triage_prompt` in `project/prompts.yml` spells the conventions out in four
lines. That alone moved the score from **15/20 to 18/20** — measured, not assumed.

**Show examples.** Few-shot: paste in worked cases and let the model infer the rule. `prompts.yml`
carries three; `--diff` scores the assistant with and without them:


In [27]:
done = subprocess.run([sys.executable, "score_triage.py", "--diff"],
                      cwd=HERE, capture_output=True, text=True, timeout=1800)
print(done.stdout or done.stderr[-1500:])


WITHOUT examples: 18/20 = 90%
  0 unusable response(s) · 1120 output tokens · $0.0028 for the run ($0.14 per 1,000 tickets)

  ticket    hand label         assistant said
  TCK-003   complaint          order_status
  TCK-005   return             warranty

  commonest confusions: complaint -> order_status (x1), return -> warranty (x1)

WITH examples: 17/20 = 85%
  0 unusable response(s) · 1123 output tokens · $0.0037 for the run ($0.19 per 1,000 tickets)

  ticket    hand label         assistant said
  TCK-002   refund             complaint
  TCK-005   return             warranty
  TCK-015   warranty           product_question

  commonest confusions: refund -> complaint (x1), return -> warranty (x1), warranty -> product_question (x1)

The examples bought -1 ticket(s) = -5 percentage points, for +34% cost.
  fixed : TCK-003
  BROKE : TCK-002, TCK-015  <- examples can cost you tickets too



### 8.3 The examples bought nothing — and that is a result

Across several runs the examples scored between −2 and 0 tickets against the baseline, for tokens
resent on **every call, forever**. So they ship **off** (`USE_FEW_SHOT = False` in `config.py`, with
the measurement written next to the flag). Notice they also *broke* a ticket the baseline got right:
examples reshape the whole decision boundary, not just the case you were looking at.

Two habits to keep:

1. **Write the rule before you show an example.** A rule costs tokens once; examples cost tokens on
   every call.
2. **"We measured it, it did not help, we did not ship it"** is a complete engineering answer.

> 🚩 **An example in your prompt is not a test case.** The first version of these examples reused
> three tickets from `tickets.yml` and scored a perfect 20/20 — because three of the twenty were
> sitting in the prompt. The shipped examples are written fresh and appear nowhere in the eval set.
> Session 9 builds a proper held-out eval set.


## 9. When the call fails

Roughly one call in a few hundred will not come back clean, and each kind of failure wants a
different reaction.

<img src="figures/four-endings.png" width="860"
     alt="One model call fanning out into its four possible endings — a usable answer, a transient
     failure, your own mistake, and a 200 OK that contains nothing usable — each with the response it
     deserves.">

| ending | looks like | what to do |
|---|---|---|
| a usable answer | `.parsed` is a validated object | use it |
| transient | 429, 500, 503, a timeout | **retry with backoff** |
| your mistake | 400, unknown model, `ValidationError` | **fix the code** — retrying a bug runs it again |
| 200 OK, nothing usable | `.parsed is None` | read `finish_reason` (section 6.3) |
| a real answer you dislike | a valid `TriageResult` you disagree with | **do not retry** — that is a business decision |

### 9.1 Errors have types. Use them.


In [28]:
# The amateur version of this is `if "429" in str(err)`. The SDK gives you classes.
try:
    client.models.generate_content(model="gemini-does-not-exist", contents="hi")
except errors.ClientError as err:                 # 4xx — our request was wrong
    print(f"ClientError  code={err.code}  status={err.status}")
    print(f"  {err.message[:110]}")
    print("\n  -> 4xx is OUR bug. No amount of retrying invents a model that does not exist.")
except errors.ServerError as err:                 # 5xx — their side
    print(f"ServerError  code={err.code} — retry this one")

ClientError  code=404  status=NOT_FOUND
  models/gemini-does-not-exist is not found for API version v1beta, or is not supported for generateContent. Cal

  -> 4xx is OUR bug. No amount of retrying invents a model that does not exist.


`ClientError` (4xx) and `ServerError` (5xx) carry `.code`, `.status` and `.message`. Branch on
those, never on substrings of an English error message.

### 9.2 The retry loop already exists

Before writing backoff by hand, check the SDK. This is `build_client()` in `project/utils.py`:

```python
genai.Client(
    http_options=types.HttpOptions(
        timeout=int(settings.REQUEST_TIMEOUT_S * 1000),          # the SDK wants milliseconds
        retry_options=types.HttpRetryOptions(
            attempts=settings.MAX_RETRIES,
            http_status_codes=[408, 429, 500, 502, 503, 504],   # transient only: no 400, no 403
        ),
    ),
)
```

It retries transient HTTP failures with exponential backoff before you ever see an exception. It does
**not** retry a truncated body, a `ValidationError`, or an answer you disagree with — those are yours.

> **Never blindly retry a call that changes something.** Triage is a read. Session 5's tools include
> actions, and a blind retry there issues the refund twice.

### 9.3 Validate, and repair

Section 6.3 left a problem open: Pydantic rejects an over-long summary and `.parsed` is `None`.
Retrying the identical call is a coin toss. Retrying **with the error attached** is not:


In [30]:
def ask_and_repair(schema, contents, attempts: int = 3):
    """Ask, validate, and on failure ask again WITH the validation error attached."""
    messages = contents
    for attempt in range(1, attempts + 1):
        r = client.models.generate_content(
            model=settings.MODEL, contents=messages,
            config=types.GenerateContentConfig(
                response_mime_type="application/json", response_schema=schema))
        try:
            return schema.model_validate_json(r.text), attempt
        except ValidationError as err:
            if attempt == attempts:
                raise
            # The repair prompt: what you produced, and precisely what was wrong with it.
            problems = "; ".join(f"{e['loc'][0]}: {e['msg']}" for e in err.errors())
            print(f"  attempt {attempt} rejected -> {problems}")
            messages = (f"{contents}\n\nYour previous answer was rejected:\n{r.text}\n\n"
                        f"It broke these rules: {problems}\nReturn a corrected version.")


try:
    result, took = ask_and_repair(Capped, block + "\n\nBe thorough and detailed in the summary.")
    print(f"\naccepted on attempt {took}: {result.summary!r} ({len(result.summary)} chars)")
except ValidationError:
    print("\nthree attempts, still rejected — which is also a result. When a repair loop cannot")
    print("converge, the schema and the model disagree structurally: relax the rule, or say it")
    print("in the description where the model can actually read it.")

  attempt 1 rejected -> summary: String should have at most 60 characters



accepted on attempt 2: 'Charged twice for a single order; request refund.' (49 chars)


Why it works: the model is told the *specific* violation, and models are much better at editing than
at guessing what upset you. Keep the attempts capped — three is plenty. If three fail, the schema and
the model disagree about something structural; say the rule in the description instead.


## 10. What does it cost?

Two calls per ticket now. Someone will ask what that means at 200 tickets a day. Prices live in
`labs/data/prices.yml`, one copy for the whole course, with a `verified:` date because they change.


In [31]:
PRICES = {p["id"]: p for p in
          yaml.safe_load((settings.DATA_DIR / "prices.yml").read_text(encoding="utf-8"))}

for p in PRICES.values():
    print(f"  {p['id']:24} in ${p['input']:<6} out ${p['output']:<7} verified {p['verified']}")

  gemini-3.1-flash-lite    in $0.25   out $1.5     verified 2026-08-30
  gemini-3.6-flash         in $0.75   out $3.75    verified 2026-08-30
  gemini-2.5-pro           in $1.25   out $10.0    verified 2026-08-30
  gemini-embedding-2       in $0.15   out $0.0     verified 2026-08-30


In [32]:
def cost(usage, model: str) -> float:
    row = PRICES[model]
    return (usage.prompt_token_count * row["input"]
            + usage.candidates_token_count * row["output"]) / 1_000_000


t0 = time.perf_counter()
r = client.models.generate_content(
    model=settings.MODEL, contents=block,
    config=types.GenerateContentConfig(
        response_mime_type="application/json", response_schema=TriageResult))
elapsed = time.perf_counter() - t0

u = r.usage_metadata
per_call = cost(u, settings.MODEL)
print(f"in {u.prompt_token_count} tok · out {u.candidates_token_count} tok "
      f"-> ${per_call:.6f} in {elapsed:.2f}s")
print(f"two calls per ticket, 200 tickets/day -> ${per_call * 2 * 200:.2f}/day "
      f"= ${per_call * 2 * 200 * 30:.2f}/month")

in 43 tok · out 58 tok -> $0.000098 in 1.19s
two calls per ticket, 200 tickets/day -> $0.04/day = $1.17/month


**Output tokens cost several times more than input tokens.** "Make the reply shorter" is a real cost
optimisation, and a one-sentence `summary` is cheaper than a paragraph.

### 10.1 Does the schema count as input tokens?

The JSON Schema is sent with every call, so you might expect to pay for it as input. Google's token
docs say system instructions and tools *are* counted as input; they say nothing about the response
schema. So measure it:


In [33]:
bare = client.models.generate_content(model=settings.MODEL, contents=block)
typed = client.models.generate_content(
    model=settings.MODEL, contents=block,
    config=types.GenerateContentConfig(
        response_mime_type="application/json", response_schema=TriageResult))

b, t = bare.usage_metadata.prompt_token_count, typed.usage_metadata.prompt_token_count
print(f"same ticket, no schema : {b} input tokens")
print(f"same ticket, TriageResult: {t} input tokens   ({t - b:+d})")
print(f"\nat 200 tickets/day that is ${(t - b) * 200 * 30 * PRICES[settings.MODEL]['input'] / 1e6:.2f}/month"
      f" for the schema alone.")

same ticket, no schema : 43 input tokens
same ticket, TriageResult: 43 input tokens   (+0)

at 200 tickets/day that is $0.00/month for the schema alone.


**Zero.** On the Gemini API today, `prompt_token_count` is the same with and without the schema — and
it stays the same with a schema ten times larger (checked with a 9,000-character one). The constraint
is applied at the decoder, not pasted into your prompt, and billing follows `usage_metadata`.

Two things this does not change: **few-shot examples are still billed** — they go into the system
instruction; look at the cost column in section 8.2 — and a huge schema still costs the model
attention and you latency. And the fact may not travel: another provider may count it. When a cost
claim matters, measure it on the provider you actually use.


## 11. Project v0.2 — the first pipeline

<img src="figures/v02-pipeline.png" width="860"
     alt="Version 0.2's pipeline: a ticket is triaged by the first model call, the answer passes
     through a schema-and-validator gate into a typed TriageResult the code branches on, and that
     result is handed to a second call that drafts the guarded reply, with both calls metered for
     cost.">

**We walk this through in the terminal.** Open `session-03/project/` in your editor and follow along —
the notebook is where you learn the idea, the editor is where the software lives.

```
labs/session-03/
├── score_triage.py        the measurement from section 8 — NOT part of the assistant
└── project/
    ├── config.py      ★   Settings(BaseSettings) — typed configuration, replaces v0.1's constants
    ├── schemas.py     ★   TriageResult + its validators — the object that survives to session 16
    ├── prompts.yml    ★   + triage_prompt, + few_shot_examples (off by default, and measured)
    ├── utils.py       ★   plumbing: build_client() · checked() · load_tickets() · cost()
    └── triage.py      ★   the program: triage() -> TriageResult -> draft_reply()
```

| file | why it is separate |
|---|---|
| `config.py` | every knob, typed and defaulted, in one readable list. `uv run python config.py` prints what it resolved to |
| `schemas.py` | the shapes, in one place. Session 10 imports this exact class; session 13 makes it an API response |
| `prompts.yml` | prompt text still never lives in Python — now it holds the triage prompt too |
| `utils.py` | the plumbing every LLM script needs — client with retries, the `.parsed is None` guard, fixtures, cost — kept apart so `triage.py` reads as the program |
| `triage.py` | the program: two calls, one feeding the other, both metered, both guarded |
| `score_triage.py` | **beside** the notebook, not inside `project/` — a measurement tool, not part of the assistant |

**Run it:**

```bash
cd labs/session-03/project
uv run python triage.py --limit 3
```

In [34]:
done = subprocess.run([sys.executable, "triage.py", "--limit", "2"],
                      cwd=HERE / "project", capture_output=True, text=True, timeout=600)
print(done.stdout[-2600:])

ShopWise Support Assistant v0.2.0 · gemini-3.1-flash-lite · 2 ticket(s)

TCK-001  order_status | medium | calm | needs_human=False
  Customer is inquiring about the delivery status of order ORD-5001.
--------------------------------------------------------------------------
Hello,

Thank you for reaching out, and I am sorry that your order has not arrived yet. I understand how concerning it is when tracking information does not update as expected.

I do not have access to courier tracking or order systems, so I am passing your ticket to a human colleague who can look into this for you. They will review your order and reply to this email with an update, usually within one working day.

ShopWise Support

  [$0.000452 this ticket · $0.000452 running]

TCK-002  refund | medium | calm | needs_human=True
  Customer requesting refund for defective headphones with crackling left earcup.
--------------------------------------------------------------------------
Hello,

I am very sorry to hear t

**The pipeline is the new thing.** `draft_reply()` receives the `TriageResult`, so the second call
already knows what kind of ticket it is answering — and when `needs_human` is true it is told, in
words, not to state an outcome. That `needs_human` came from the model *and* from the validator in
`schemas.py`, which has the final word: **the guardrail is enforced in code, not hoped for in a
prompt.**

v0.2's limits, each a future session:

- **Knows no ShopWise facts** — it still cannot look up ORD-5001 → **S05** tools, **S08** retrieval
- **Guardrails still mostly soft** — one validator is not a policy layer → **S07**
- **One provider, hand-written** — every call is Gemini-shaped → **S04**


## 12. Recap, and the hook

| v0.1 | v0.2 |
|---|---|
| Prose replies only | A typed `TriageResult` your code branches on |
| Configuration was five `os.getenv` calls | Validated `Settings`, failing at startup with the field name |
| Guardrails lived in the prompt | One of them is a validator the model cannot talk out of |
| No failure handling | Four endings, four responses, and a repair loop |
| Cost unknown | Per-ticket cost and a per-month projection |
| Prompt quality was an opinion | **Triage accuracy is a number**, with its ceiling named |

**What you can now do:** write a Pydantic model with constraints, validators and descriptions, and
know which of those the provider enforces; explain what constrained decoding does; recognise the four
ways a call ends and recover from the one that looks like success; score a prompt against ground
truth; and say "we measured it, it did not help, we did not ship it."

### The hook

> You show the cost report. Your CTO says: *"Good. Leadership wants a Plan B vendor by Friday, in case
> Google raises prices. How long to switch?"*

Every call says `client.models.generate_content`. Every schema goes in through `response_schema`.
Every error class is Google's, and section 6.4 showed the *next* provider accepts a different subset
of JSON Schema. **All of it is Gemini-shaped**, and swapping vendors means rewriting all of it.

Next session: one interface, four providers, the same code.

**Before session 4:** `exercises.ipynb`. The ⭐⭐⭐ asks you to build an *objective* metric — extraction
accuracy — to sit beside the conventional one from section 8.1.


---

# 🎓 Advanced appendix

## A. Nested schemas and lists

Fields can be models, and a response schema can be a list. This is how you extract *many things*
rather than one — and it is the shape session 5's tools consume.

In [35]:
class LineItem(BaseModel):
    sku: str = Field(description="SKU-XXX format, exactly as written in the text")
    quantity: int = Field(ge=1)


class OrderMention(BaseModel):
    order_ids: list[str] = Field(description="Every ORD-XXXX in the text; empty list if none")
    items: list[LineItem] = Field(default_factory=list)


r = client.models.generate_content(
    model=settings.MODEL,
    contents="I want to return 2 of SKU-300 from order ORD-5002, and cancel ORD-5007 entirely.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json", response_schema=OrderMention))
print(r.parsed)

# A bare list works as a response schema too — no wrapper object needed.
r2 = client.models.generate_content(
    model=settings.MODEL, contents="2 of SKU-300 and 1 of SKU-411, please.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json", response_schema=list[LineItem]))
print(r2.parsed, "<-", type(r2.parsed).__name__)

order_ids=['ORD-5002', 'ORD-5007'] items=[LineItem(sku='SKU-300', quantity=2)]


[LineItem(sku='SKU-300', quantity=2), LineItem(sku='SKU-411', quantity=1)] <- list


**Empty list, not `None`.** Say so in the description. A `None` forces every consumer to check
before iterating; an empty list just works — and this exact field feeds the tools you build in session
5, where the consumer is a `for` loop that looks each ID up.

Keep the nesting shallow. One level is fine everywhere; three levels is a schema that some provider
will reject and every model will lose its place in.

## B. `TypeAdapter` — validation without a model class

Not everything you validate deserves a class. `TypeAdapter` gives you the whole Pydantic machinery for
any type annotation, which is the tool for a bare list, a dict of known shape, or a value you just
pulled out of YAML.

In [36]:
from pydantic import TypeAdapter

Categories = TypeAdapter(list[Literal["billing", "refund", "shipping"]])
print(Categories.validate_python(["billing", "refund"]))

try:
    Categories.validate_python(["billing", "Billing"])      # capital B is not in the list
except ValidationError as err:
    e = err.errors()[0]
    print(f"  rejected at index {e['loc']}: {e['msg'][:60]}")

# It is also the cleanest way to see the JSON Schema of a plain annotation:
print("\nschema:", json.dumps(TypeAdapter(list[int]).json_schema()))

['billing', 'refund']
  rejected at index (1,): Input should be 'billing', 'refund' or 'shipping'

schema: {"items": {"type": "integer"}, "type": "array"}


Use it to validate the fixtures your own program loads. `tickets.yml` is hand-edited; a
`TypeAdapter(list[Ticket])` over it at startup turns "somebody typo'd a category three weeks ago" into
an error on the commit that did it.

## C. `Enum` or `Literal`?

Both close the value space. They differ in what the rest of your program gets to do with the value.

| | `Literal["billing", ...]` | `class Category(str, Enum)` |
|---|---|---|
| the value is | a plain `str` | an enum member (`Category.BILLING`) |
| comparing | `cat == "billing"` | `cat is Category.BILLING` |
| refactoring | grep for a string | your IDE renames it |
| attaching behaviour | nowhere to put it | methods on the enum |
| JSON Schema | `enum: [...]` | `enum: [...]` — identical on the wire |

This course uses `Literal` because it is one line, reads like the data, and a beginner can see the
whole value space without opening another file. Reach for an `Enum` when the values need behaviour
attached — an SLA per urgency level, a queue per category — or when the same set is used in twenty
places and you want the rename to be safe.

In [ ]:
from enum import Enum


class Urgency(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

    @property 
    def sla_hours(self) -> int:            # behaviour lives WITH the value
        return {"low": 72, "medium": 24, "high": 4}[self.value]


class Escalation(BaseModel):
    urgency: Urgency


e = Escalation(urgency="high")             # a plain string still validates
print(e.urgency, "| respond within", e.urgency.sla_hours, "hours")
print("on the wire it is identical:", Escalation.model_json_schema()["$defs"]["Urgency"]["enum"])

Urgency.HIGH | respond within 4 hours
on the wire it is identical: ['low', 'medium', 'high']


## D. Raw JSON Schema, without Pydantic

`response_json_schema` takes a plain dict, for when you are generating schemas dynamically — from a
database table, a user-defined form — or working in a codebase that does not use Pydantic.

```python
config=types.GenerateContentConfig(
    response_mime_type="application/json",
    response_json_schema={
        "type": "object",
        "properties": {"category": {"type": "string", "enum": ["billing", "shipping"]}},
        "required": ["category"],
    })
```

You lose validation on the way back in — you get a dict, and whatever is wrong with it is now your
problem. Which is the argument for Pydantic in one sentence: it is the same schema, plus the check.

## E. Which model should triage?

The stronger model costs 3× the input and 2.5× the output. Does it triage better? **Measure it, do not
assume it** — that is the whole session.

In [38]:
hits = {}
for tier in (settings.MODEL, settings.MODEL_STRONG):
    right = 0
    for t in TICKETS[:8]:                       # 8 tickets to keep the appendix quick
        got = client.models.generate_content(
            model=tier,
            contents=f"Triage this ticket.\n\nSubject: {t['subject']}\n{t['body']}",
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=TriageResult)).parsed
        right += got is not None and got.category == t["category"]
    hits[tier] = right
    print(f"  {tier:24} {right}/8")

print("\nOn a sample this small the difference is inside the noise — which is itself")
print("the lesson: 8 tickets cannot separate two models, and neither can 20.")
print("Section 7.2 already showed this set moving by 3 between identical runs.")

  gemini-3.1-flash-lite    7/8


  gemini-3.6-flash         7/8

On a sample this small the difference is inside the noise — which is itself
the lesson: 8 tickets cannot separate two models, and neither can 20.
Section 7.2 already showed this set moving by 3 between identical runs.


## F. The same idea on other providers — a preview of session 4's problem

Every provider supports this. No two spell it the same way.

| | Gemini | OpenAI | Anthropic |
|---|---|---|---|
| how you ask | `response_schema=` | `response_format=` with a JSON-Schema wrapper | a tool the model must call |
| what you read | `.parsed` | `.output_parsed` | the tool-call arguments |
| tokens | `usage_metadata.prompt_token_count` | `usage.input_tokens` | `usage.input_tokens` |
| errors | `ClientError` / `ServerError` | `APIStatusError` subclasses | `APIStatusError` subclasses |
| schema subset | no `additionalProperties` | requires `additionalProperties: false` | broadly permissive |

Look at row four of that table twice. The two most popular providers disagree about a single JSON
Schema keyword in **exactly opposite directions** — Gemini rejects the schema Pydantic's
`extra="forbid"` produces, and OpenAI's strict mode requires it. There is no schema that satisfies
both, which means there is no "just swap the base URL" version of this problem.

That is next session's whole subject, and now you know precisely why it is not a five-minute job.

## G. Does a "reasoning first" field improve accuracy? Measure it

Folklore says: declare a `reasoning` field *before* `category`, so the model thinks before it decides.
Section 7.1 shows the mechanism is real — the field really is written first. Whether it *helps* is a
separate question, and `score_triage.py --schemas` answers it: it scores the shipped schema, a
reasoning-first variant, and a reasoning-*last* control (same field, same description, declared last,
so it costs the same tokens but cannot influence the answer).

```bash
uv run python score_triage.py --schemas
```

Across six authoring runs: shipped **17–18/20** · reasoning-first **15–18/20** · reasoning-last
**17–18/20**, with reasoning-first costing about 85% more output tokens. It never once came out ahead.
The misses here are convention disagreements, and inviting the model to reason lets it re-derive a
rule of its own instead of following ours. Reasoning-first earns its keep on tasks that need
multi-step work; session 5 uses it there, and measures it there.

Two cautions: twenty tickets move by a point between identical runs, so a one-ticket difference is
noise; and always run the control arm, or you cannot tell "reasoning helped" from "a longer answer
helped".


## H. A default does not make a field private

```python
class HasDefault(BaseModel):
    category: Literal["billing", "refund", "other"]
    internal_note: str = "set by our code later"     # optional in the schema — but still IN it
```

The model can see `internal_note` and will fill it in with something plausible. Three months later
nobody knows where that string came from. **If the model should not fill a field, it does not belong
in the response schema.** Keep the model's answer and your own bookkeeping in separate classes and
combine them in code:

```python
result = TriageResult.model_validate_json(r.text)        # what the model said
record = TicketRecord(triage=result, received_at=now())  # what we know
```
